# 06 · Registro de los experimentos del perfil v02 en MLflow

**Objetivo:** dejar registrados en MLflow los experimentos del notebook 05 (barrido de K, alternativas de entrada, estabilidad) y el modelo elegido, para poder compararlos y defenderlos.

**Orden:** después del 05. Este notebook **no recalcula nada**: lee `experimentos_v02.csv` y `modelo_elegido_v02.json`, que guarda el 05, y los registra. Si el 07 ya se ha ejecutado, adjunta también los ficheros del perfil.

Los registros van a `mlruns/` (ignorada por Git), en una base SQLite. Se puede cambiar la carpeta con la variable de entorno `SITEWISE_MLRUNS`. Para verlos: `mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db`.

In [1]:
import json, os
from pathlib import Path

import mlflow
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
ML = ROOT / 'data/processed/sitewise_ml'
REFERENCIA = ROOT / 'data/reference'

# Comprobación de orden: el 05 debe haberse ejecutado.
for f in ('experimentos_v02.csv', 'modelo_elegido_v02.json'):
    if not (ML / f).exists():
        raise FileNotFoundError(f'Falta {f}: ejecuta antes el notebook 05.')
experimentos = pd.read_csv(ML / 'experimentos_v02.csv')
modelo = json.loads((ML / 'modelo_elegido_v02.json').read_text(encoding='utf-8'))
print(len(experimentos), 'pruebas | modelo elegido:', modelo['k'], 'grupos con', ', '.join(modelo['entradas']))

# MLflow 3 no admite la carpeta ./mlruns como almacén: se usa una base SQLite dentro de mlruns/.
MLRUNS = Path(os.environ.get('SITEWISE_MLRUNS', ROOT / 'mlruns'))
MLRUNS.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri('sqlite:///' + (MLRUNS / 'mlflow.db').as_posix())
NOMBRE_EXPERIMENTO = 'sitewise-perfil-v02'
if mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO) is None:
    mlflow.create_experiment(NOMBRE_EXPERIMENTO, artifact_location=(MLRUNS / 'artifacts').as_uri())
mlflow.set_experiment(NOMBRE_EXPERIMENTO)
print('MLflow', mlflow.__version__, '| experimento:', NOMBRE_EXPERIMENTO)

d:\HELEN\PERSONAL\FORMACIÓN\0-ia_formacion\BOOTCAMP - F5\CURSO\PROYECTO_FINAL\SiteWise\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


15 pruebas | modelo elegido: 4 grupos con idx_logistica, idx_hidrica, idx_conectividad
MLflow 3.18.0 | experimento: sitewise-perfil-v02


## 1. Registrar el barrido de K y las alternativas de entrada

Una ejecución de MLflow por cada fila de `experimentos_v02.csv`, con sus parámetros y métricas.

In [2]:
METRICAS = ['inercia', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'grupo_menor_pct', 'cobertura_pct', 'ari_medio', 'ari_minimo']
for _, f in experimentos.iterrows():
    with mlflow.start_run(run_name=f"kmeans_{f['configuracion']}_k{int(f['k'])}_{f['tipo']}"):
        mlflow.set_tags({'algoritmo': 'kmeans', 'tipo': f['tipo'], 'version_modelo': modelo['version_modelo']})
        mlflow.log_params({'configuracion': f['configuracion'], 'variables': f['variables'], 'usa_pca': False, 'filas': int(f['filas']),
                           'k': int(f['k']), 'semilla': int(f['semilla']), 'n_init': int(f['n_init']), 'sha256_dataset': f['sha256_dataset']})
        mlflow.log_metrics({m: float(f[m]) for m in METRICAS if pd.notna(f.get(m))})
print(experimentos['tipo'].value_counts().to_string())

2026/11/28 11:05:46 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/11/28 11:05:46 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

tipo
alternativa_entrada    8
barrido_k              7


## 2. Registrar el modelo elegido

Se registra con los parámetros de `modelo_elegido_v02.json` y las métricas de su fila en el barrido. Si ya existen los ficheros del perfil (notebook 07), se adjuntan.

In [3]:
elegido = experimentos[(experimentos['tipo'] == 'barrido_k') & (experimentos['k'] == modelo['k'])].iloc[0]
with mlflow.start_run(run_name='perfil_v02_final'):
    mlflow.set_tags({'algoritmo': 'kmeans', 'tipo': 'modelo_final', 'version_modelo': modelo['version_modelo']})
    mlflow.log_params({'configuracion': 'logistica_hidrica_conectividad', 'variables': ', '.join(modelo['entradas']), 'usa_pca': False,
                       'filas': int(elegido['filas']), 'k': modelo['k'], 'semilla': modelo['semilla'], 'n_init': modelo['n_init'],
                       'sha256_dataset': elegido['sha256_dataset'], 'motivo_k': modelo['motivo'],
                       'huecos_hidrica': 'media de los 5 municipios cercanos con dato', 'huecos_logistica': 'sin imputar'})
    mlflow.log_metrics({m: float(elegido[m]) for m in METRICAS if pd.notna(elegido.get(m))})
    adjuntos = [f for f in ('perfiles_municipios_v02.csv', 'perfiles_definicion_v02.csv', 'perfiles_estado_v02.csv') if (REFERENCIA / f).exists()]
    for f in adjuntos:
        mlflow.log_artifact(str(REFERENCIA / f))
print('Modelo registrado. Ficheros adjuntos:', adjuntos or 'ninguno (ejecuta el 07 y repite esta celda)')

Modelo registrado. Ficheros adjuntos: ['perfiles_municipios_v02.csv', 'perfiles_definicion_v02.csv', 'perfiles_estado_v02.csv']
